# Stage 03 – Mistral-7B QLoRA 4-bit cho Kaggle

Mục tiêu: giữ Mistral-7B-Instruct-v0.2 theo bài gốc nhưng chạy được trên GPU Kaggle 14-16GB.

Dùng: bitsandbytes 4-bit NF4 + PEFT LoRA + transformers.Trainer.
Không dùng: trl/SFTTrainer để giữ custom loss mask cho target report.

In [ ]:
# Kaggle cell 1: dependencies
# Transformers 5.0 yêu cầu bitsandbytes >= 0.46.1 cho 4-bit quantization.
# Dùng subprocess để lỗi pip không bị nuốt im lặng.
try:
    get_ipython  # noqa: F821
    import subprocess, sys
    pkgs = [
        "bitsandbytes>=0.46.1",
        "evaluate>=0.4.3",
        "rouge-score>=0.1.2",
        "sacrebleu>=2.4.3",
        "sentencepiece>=0.2.0",
    ]
    print("Installing/checking:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pkgs])
    print("Dependency install/check finished.")
except NameError:
    pass

In [ ]:
# CELL 2 — Kiểm tra version trước khi chạy
import sys
packages = [
    "numpy", "pandas", "torch",
    "transformers", "peft", "accelerate", "bitsandbytes",
    "datasets", "evaluate",
]
for pkg in packages:
    try:
        m = __import__(pkg)
        print(pkg, getattr(m, "__version__", "?"), "OK")
    except Exception as e:
        print(pkg, "ERROR:", repr(e))

import torch
print("\ncuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# CELL 3 — Imports + config

import gc, json, math, os, random, re, time, inspect
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

try:
    import evaluate as hf_evaluate
except Exception:
    hf_evaluate = None

# ======== CONFIG ========
SEED = 391
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

RUN_MODE = "full"          # full | smoke | eval_only

MODEL_NAME_OR_PATH = os.environ.get(
    "MODEL_NAME_OR_PATH",
    "mistralai/Mistral-7B-Instruct-v0.2"
)
USE_FLASH_ATTN = False     # đổi True nếu GPU A100/H100 hỗ trợ flash_attn

CACHE_ROOT_HINTS = [
    Path("/kaggle/input/notebooks/tundng111/vimedpet-01-build-25d-cache-streaming/vimedpet_q1_outputs/01_cache"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_mistral_qlora_outputs")
ADAPTER_DIR  = OUTPUT_ROOT / "mistral_lora_adapter"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Sequence length theo Report 2: 512 phủ ~98.74% report Part 3
MAX_SEQ_LENGTH         = 512
PER_DEVICE_BATCH_SIZE  = 1
GRAD_ACCUM             = 8
LEARNING_RATE          = 2e-4
NUM_EPOCHS             = 1
MAX_STEPS_SMOKE        = 30
MAX_TRAIN_SMOKE        = 64
MAX_EVAL_SMOKE         = 16
EVAL_SAMPLES_FULL      = 64   # None = đánh giá toàn validation

LORA_R                 = 8
LORA_ALPHA             = 16
LORA_DROPOUT           = 0.05
TARGET_MODULES         = ["q_proj","k_proj","v_proj","o_proj",
                           "gate_proj","up_proj","down_proj"]

USE_VISUAL_SUMMARY     = True
GEN_MAX_NEW_TOKENS     = 384
GEN_TEMPERATURE        = 0.2
GEN_TOP_P              = 0.9

print("RUN_MODE   :", RUN_MODE)
print("MODEL      :", MODEL_NAME_OR_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))

In [ ]:
# CELL 4 — Locate Stage-01 cache
def find_file(name: str, hints: List[Path]) -> Path:
    hits = []
    for h in hints:
        if not h.exists(): continue
        direct = h / name
        if direct.exists(): hits.append(direct)
        hits.extend(h.rglob(name))
    hits = sorted(set(hits), key=lambda p: len(str(p)))
    if not hits:
        raise FileNotFoundError(f"Cannot find {name} under: {hints}")
    return hits[0]

cache_meta_path  = find_file("q1_cache_meta.json", CACHE_ROOT_HINTS)
cache_root       = cache_meta_path.parent
cache_index_path = cache_root / "q1_cache_index.csv"
manifest_path    = cache_root / "q1_clean_manifest_for_cache.csv"
meta             = json.loads(cache_meta_path.read_text(encoding="utf-8"))

# Stage 01 hiện tại lưu đường dẫn tuyệt đối trong `cache_path`; bản cũ có thể dùng `mmap_path`.
memmap_meta_path = meta.get("cache_path") or meta.get("mmap_path")
assert memmap_meta_path, f"Missing cache_path/mmap_path in meta: {meta.keys()}"
memmap_path      = Path(memmap_meta_path)
if not memmap_path.exists():
    memmap_path = cache_root / Path(memmap_meta_path).name

assert cache_index_path.exists(), f"Missing: {cache_index_path}"
assert manifest_path.exists(),    f"Missing: {manifest_path}"
assert memmap_path.exists(),      f"Missing: {memmap_path}"

print("cache_root  :", cache_root)
print("memmap_path :", memmap_path)
print("cache meta  :", json.dumps(meta, ensure_ascii=False, indent=2))

In [ ]:
# CELL 5 — Load manifest + join cache index
manifest    = pd.read_csv(manifest_path)
cache_index = pd.read_csv(cache_index_path)

# Stage 01 schema mới dùng status/error; schema cũ có thể dùng cache_ok/cache_error.
if "cache_ok" not in cache_index.columns:
    assert "status" in cache_index.columns, f"Missing status/cache_ok in cache index: {cache_index.columns.tolist()}"
    cache_index["cache_ok"] = cache_index["status"].astype(str).str.lower().eq("ok")
if "cache_error" not in cache_index.columns:
    cache_index["cache_error"] = cache_index["error"] if "error" in cache_index.columns else ""

if cache_index["cache_ok"].dtype != bool:
    cache_index["cache_ok"] = (
        cache_index["cache_ok"].astype(str).str.lower().isin(["true","1","yes","ok"])
    )

join_cols = (
    ["case_id","row_index"]
    if "row_index" in manifest.columns and "row_index" in cache_index.columns
    else ["case_id"]
)
cache_join_cols = [c for c in ["case_id","row_index","cache_ok","cache_error","cache_offset","status","ct_shape","pet_shape"] if c in cache_index.columns]
df = manifest.merge(
    cache_index[cache_join_cols],
    on=join_cols, how="inner", suffixes=("","_cache")
)
target_col = next(
    (c for c in ["model_target_text","report_text_clean","target_text","report_text"] if c in df.columns),
    None
)
assert target_col, f"No target column. Available: {df.columns.tolist()}"
df[target_col] = df[target_col].fillna("").astype(str)
df = df[(df["cache_ok"]==True) & (df[target_col].str.len()>10)].copy()
split_col = "clean_split" if "clean_split" in df.columns else "split"
assert split_col in df.columns

expected_rows = int(meta.get("ok_rows") or meta.get("n_cache_rows") or meta.get("n_cases_manifest") or len(manifest))
assert len(df) == expected_rows, f"Joined rows mismatch: df={len(df)} expected={expected_rows}. Check Stage 01 schema/input."

print("joined df   :", df.shape)
print(df[split_col].value_counts().to_string())
display(df.head(2))

In [ ]:
# CELL 6 — Open memmap + visual summary
shape_meta = meta.get("shape")
if isinstance(shape_meta, list) and len(shape_meta) == 4:
    n = int(shape_meta[0])
    channels = int(shape_meta[1])
    image_size_h = int(shape_meta[2])
    image_size_w = int(shape_meta[3])
    assert image_size_h == image_size_w, f"Expected square cache image, got shape={shape_meta}"
    image_size = image_size_h
else:
    n = int(meta.get("n") or meta.get("n_cases_manifest") or len(cache_index))
    channels_meta = meta.get("channels", 3)
    channels = len(channels_meta) if isinstance(channels_meta, list) else int(channels_meta)
    image_size = int(meta.get("image_size", 224))
mm = np.memmap(memmap_path, dtype=np.uint8, mode="r",
               shape=(n, channels, image_size, image_size))
assert (n, channels, image_size, image_size) == tuple(mm.shape)
assert channels == 3, f"Stage 03 expects Stage 01 3-channel cache, got channels={channels}"
print("memmap shape:", mm.shape)

_VIS_CACHE: Dict[int,str] = {}

def visual_summary(row_index: int) -> str:
    if row_index in _VIS_CACHE:
        return _VIS_CACHE[row_index]
    arr      = np.asarray(mm[int(row_index)], dtype=np.float32) / 255.0
    # Stage 01 chính thức là 3 kênh: CT central, PET MIP, overlay.
    if arr.shape[0] >= 3:
        ct_like = arr[:1]
        pet_like = arr[1:2]
        overlay_like = arr[2:3]
    else:
        ct_like = arr[:1]
        pet_like = arr[-1:]
        overlay_like = arr[-1:]
    pet_p95  = float(np.percentile(pet_like, 95))
    pet_hot  = float((pet_like > max(0.65, pet_p95)).mean())
    ct_hi    = float((ct_like > 0.75).mean())
    overlay_mean = float(overlay_like.mean())
    left     = float(pet_like[:, :, :image_size//2].mean())
    right    = float(pet_like[:, :, image_size//2:].mean())
    asym     = abs(left - right) / max((left + right)/2, 1e-6)
    ch_means = ", ".join(f"c{i}={arr[i].mean():.3f}" for i in range(arr.shape[0]))
    text = (
        "TÓM TẮT ĐỊNH LƯỢNG ẢNH 2.5D PET/CT (đặc trưng phụ trợ, không thay thế đọc ảnh):\n"
        f"- PET p95={pet_p95:.3f}, hotspot_ratio={pet_hot:.4f}.\n"
        f"- CT high_density_ratio={ct_hi:.4f}.\n"
        f"- Overlay mean={overlay_mean:.3f}.\n"
        f"- Left-right PET asymmetry={asym:.3f}.\n"
        f"- Channel means: {ch_means}."
    )
    _VIS_CACHE[row_index] = text
    return text

# Quick test
print(visual_summary(int(df.iloc[0]["row_index"])))

In [ ]:
# CELL 7 — Prompt / text formatting
def clean_text(x: Any) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return re.sub(r"\s+", " ", str(x).replace("\u00a0"," ")).strip()

def clinical_info(row: pd.Series) -> str:
    parts = []
    for label, col in [
        ("Chỉ định","indication"),
        ("Bệnh sử","clinical_history"),
        ("Giới tính","sex"),
        ("Năm sinh","birth_year"),
        ("Nguồn","source_part"),
    ]:
        if col in row.index:
            v = clean_text(row[col])
            if v: parts.append(f"- {label}: {v}")
    return "\n".join(parts) if parts else "- Không có thông tin lâm sàng bổ sung."

def build_instruction(row: pd.Series) -> str:
    vis = visual_summary(int(row["row_index"])) if USE_VISUAL_SUMMARY else ""
    vis_block = f"\n{vis}\n" if vis else ""
    return (
        "Bạn là bác sĩ y học hạt nhân có kinh nghiệm đọc PET/CT. "
        "Hãy viết báo cáo PET/CT bằng tiếng Việt, văn phong lâm sàng, ngắn gọn và nhất quán.\n\n"
        "THÔNG TIN LÂM SÀNG:\n"
        f"{clinical_info(row)}"
        f"{vis_block}\n"
        "YÊU CẦU:\n"
        "- Sinh báo cáo gồm đúng hai mục: MÔ TẢ HÌNH ẢNH và NHẬN ĐỊNH KẾT QUẢ.\n"
        "- Không bịa số SUVmax nếu thông tin không đủ rõ.\n"
        "- Nếu không chắc, diễn đạt thận trọng theo phong cách báo cáo y khoa."
    ).strip()

def format_train_text(row: pd.Series) -> str:
    instr  = build_instruction(row)
    target = clean_text(row[target_col])
    if "MÔ TẢ" not in target.upper() and "NHẬN ĐỊNH" not in target.upper():
        target = "MÔ TẢ HÌNH ẢNH:\n" + target + "\n\nNHẬN ĐỊNH KẾT QUẢ:\n"
    return f"<s>[INST] {instr} [/INST]\n{target}</s>"

def format_prompt(row: pd.Series) -> str:
    return f"<s>[INST] {build_instruction(row)} [/INST]\n"

# Split
train_df = df[df[split_col]=="train"].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df   = df[df[split_col]=="validation"].reset_index(drop=True)
test_df  = df[df[split_col]=="test"].reset_index(drop=True)

if RUN_MODE == "smoke":
    train_df = train_df.head(MAX_TRAIN_SMOKE).copy()
    val_df   = val_df.head(MAX_EVAL_SMOKE).copy()

print("train/val/test:", len(train_df), len(val_df), len(test_df))

train_texts = [format_train_text(r) for _, r in train_df.iterrows()]
val_texts   = [format_train_text(r) for _, r in val_df.head(min(len(val_df), 64)).iterrows()]
print("\nSample train text (first 800 chars):")
print(train_texts[0][:800])

with (OUTPUT_ROOT / "sample_prompts.jsonl").open("w", encoding="utf-8") as f:
    for i, t in enumerate(train_texts[:10]):
        f.write(json.dumps({"i": i, "text": t}, ensure_ascii=False) + "\n")

In [ ]:
# CELL 8 — Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME_OR_PATH,
    use_fast=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("vocab size:", tokenizer.vocab_size)
print("pad/eos:", tokenizer.pad_token_id, tokenizer.eos_token_id)

# Tokenize
# Medical-grade SFT: mask phần prompt/instruction để loss chỉ học target report.
def tokenize_fn(examples):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []
    truncated_flags = []

    for text in examples["text"]:
        encoded = tokenizer(
            text,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=False,
        )
        ids = encoded["input_ids"]
        labels = ids.copy()

        marker = "[/INST]"
        if marker in text:
            prompt_prefix = text.split(marker, 1)[0] + marker + "\n"
            prefix_ids = tokenizer(
                prompt_prefix,
                add_special_tokens=False,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            )["input_ids"]
            prompt_len = min(len(prefix_ids), len(labels))
            labels[:prompt_len] = [-100] * prompt_len
        else:
            # Fallback an toàn: nếu format lỗi, không mask để không tạo all -100 labels.
            prompt_len = 0

        # Nếu target bị truncate hết, giữ lại token cuối để tránh loss NaN.
        if all(x == -100 for x in labels) and labels:
            labels[-1] = ids[-1]

        input_ids_batch.append(ids)
        attention_mask_batch.append(encoded["attention_mask"])
        labels_batch.append(labels)
        truncated_flags.append(len(tokenizer(text, add_special_tokens=False)["input_ids"]) > MAX_SEQ_LENGTH)

    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
        "labels": labels_batch,
        "was_truncated": truncated_flags,
    }

train_ds = Dataset.from_dict({"text": train_texts}).map(
    tokenize_fn, batched=True, remove_columns=["text"]
)
eval_ds = Dataset.from_dict({"text": val_texts}).map(
    tokenize_fn, batched=True, remove_columns=["text"]
) if val_texts else None

print("train_ds:", train_ds)
print("eval_ds :", eval_ds)
train_trunc_rate = float(np.mean(train_ds["was_truncated"])) if len(train_ds) else 0.0
eval_trunc_rate = float(np.mean(eval_ds["was_truncated"])) if eval_ds and len(eval_ds) else 0.0
print(f"truncation rate train={train_trunc_rate:.3f}, eval={eval_trunc_rate:.3f} at MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")

In [ ]:
# CELL 9 — Load Mistral-7B QLoRA 4-bit
print("Loading model:", MODEL_NAME_OR_PATH)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    # BUG FIX: Đặt CUDA_VISIBLE_DEVICES trước mọi thứ để tránh Trainer khởi DataParallel.
    # DataParallel + bitsandbytes 4bit gây CUBLAS_STATUS_EXECUTION_FAILED trên T4.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    free_b, total_b = torch.cuda.mem_get_info()
    print("GPU memory before load: free/total GB =", round(free_b/1e9, 2), "/", round(total_b/1e9, 2))

gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
# Tesla T4 không có bf16 tensor cores thực sự; ép fp16 để bitsandbytes ổn định.
if "t4" in gpu_name:
    COMPUTE_DTYPE = torch.float16
else:
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("Using 4-bit compute dtype:", COMPUTE_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_OR_PATH,
    quantization_config=bnb_config,
    device_map="auto",             # auto nhưng không sử dụng DataParallel vì đã đặt CUDA_VISIBLE_DEVICES=0.
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "eager",
)
model.config.use_cache = False

# Chuẩn bị model k-bit cho training: cast norm ổn định, enable input grads, checkpointing.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
    inference_mode=False,
)

# Kaggle có thể cài torchao cũ; ta không dùng torchao nên tắt dispatcher để PEFT không đụng vào.
try:
    import peft.import_utils as peft_import_utils
    import peft.tuners.lora.torchao as peft_lora_torchao
    peft_import_utils.is_torchao_available = lambda: False
    peft_lora_torchao.is_torchao_available = lambda: False
    print("Forced disable PEFT torchao dispatcher for Kaggle compatibility.")
except Exception as exc:
    print("PEFT torchao dispatcher patch skipped:", repr(exc))

model = get_peft_model(model, peft_config)

def assert_no_meta_tensors(module):
    meta_params = [name for name, p in module.named_parameters() if getattr(p, "device", None) is not None and p.device.type == "meta"]
    meta_buffers = [name for name, b in module.named_buffers() if getattr(b, "device", None) is not None and b.device.type == "meta"]
    if meta_params or meta_buffers:
        raise RuntimeError(
            "Model still has meta tensors after QLoRA load; cannot train safely. "
            f"meta_params={meta_params[:10]}, meta_buffers={meta_buffers[:10]}"
        )

assert_no_meta_tensors(model)
model.print_trainable_parameters()
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print("GPU memory after QLoRA load: free/total GB =", round(free_b/1e9, 2), "/", round(total_b/1e9, 2))

In [ ]:
# CELL 10 — Train (custom loop, không dùng Trainer để tránh DataParallel+bitsandbytes conflict)
from torch.utils.data import DataLoader
import math

def collate_sft_features(features):
    """Pad input_ids/attention_mask/labels nhưng giữ labels=-100 đã mask prompt."""
    batch = tokenizer.pad(
        [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    max_len = batch["input_ids"].shape[1]
    padded_labels = []
    for f in features:
        labels = list(f["labels"])
        pad_len = max_len - len(labels)
        padded_labels.append(labels + [-100] * pad_len)
    batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return batch

# Optimizer: paged_adamw_8bit từ bitsandbytes
import bitsandbytes as bnb
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
optimizer = bnb.optim.PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

train_loader = DataLoader(
    train_ds,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_sft_features,
    num_workers=0,
    pin_memory=False,
)

# Warmup scheduler
total_steps = MAX_STEPS_SMOKE if RUN_MODE == "smoke" else math.ceil(len(train_loader) * NUM_EPOCHS / GRAD_ACCUM)
warmup_steps_n = 3 if RUN_MODE == "smoke" else 10
def lr_lambda(step):
    if step < warmup_steps_n:
        return float(step) / max(warmup_steps_n, 1)
    progress = float(step - warmup_steps_n) / max(total_steps - warmup_steps_n, 1)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

if RUN_MODE != "eval_only":
    model.train()
    print("Starting training (custom loop, no DataParallel)...")
    optimizer.zero_grad()
    global_step = 0
    accum_loss = 0.0
    accum_n = 0
    t0 = time.time()

    epochs = 1 if RUN_MODE == "smoke" else NUM_EPOCHS
    for epoch in range(epochs):
        for local_step, batch in enumerate(train_loader, 1):
            if RUN_MODE == "smoke" and global_step >= MAX_STEPS_SMOKE:
                break
            # Move batch sang GPU
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)
            with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=True):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            accum_loss += loss.item() * GRAD_ACCUM
            accum_n += 1
            if local_step % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
                if global_step % 10 == 0 or global_step <= 3:
                    elapsed = (time.time() - t0) / 60
                    print(
                        f"  step={global_step}/{total_steps} "
                        f"loss={accum_loss/max(accum_n,1):.4f} "
                        f"lr={scheduler.get_last_lr()[0]:.2e} "
                        f"elapsed={elapsed:.1f}min"
                    )
                    accum_loss = 0.0; accum_n = 0
        else:
            continue
        break

    # Lưu adapter
    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print("Adapter saved:", ADAPTER_DIR)
    print(f"Total steps: {global_step}")
else:
    print("eval_only: skip training")

(OUTPUT_ROOT / "train_config.json").write_text(json.dumps({
    "run_mode": RUN_MODE,
    "model": MODEL_NAME_OR_PATH,
    "cache_root": str(cache_root),
    "target_col": target_col,
    "split_col": split_col,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_seq_length_basis": "Report 2 EDA: 512-token proxy covers ~98.74% Part 3 reports.",
    "train_truncation_rate": train_trunc_rate,
    "eval_truncation_rate": eval_trunc_rate,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "compute_dtype": str(COMPUTE_DTYPE),
    "quantization": "4bit_nf4_double_quant",
    "use_visual_summary": USE_VISUAL_SUMMARY,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
}, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# CELL 11 — Generation + eval
model.eval()

def generate_one(row: pd.Series) -> str:
    prompt = format_prompt(row)
    ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=(GEN_TEMPERATURE > 0),
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("[/INST]", 1)[1].strip() if "[/INST]" in decoded else decoded.strip()

gen_df = val_df.copy()
if RUN_MODE == "smoke":
    gen_df = gen_df.head(MAX_EVAL_SMOKE)
elif EVAL_SAMPLES_FULL is not None:
    gen_df = gen_df.head(EVAL_SAMPLES_FULL)

records = []
t0 = time.time()
for i, (_, row) in enumerate(gen_df.iterrows(), 1):
    pred = generate_one(row)
    ref  = clean_text(row[target_col])
    records.append({
        "case_id": row["case_id"],
        "split": row[split_col],
        "source_part": row.get("source_part",""),
        "row_index": int(row["row_index"]),
        "reference_report": ref,
        "generated_report": pred,
    })
    if i % 10 == 0:
        print(f"  {i}/{len(gen_df)} elapsed {(time.time()-t0)/60:.1f}min")

generated = pd.DataFrame(records)
generated.to_csv(OUTPUT_ROOT / "generated_reports.csv", index=False, encoding="utf-8-sig")
print("Saved generated_reports.csv:", len(generated))

In [ ]:
# CELL 12 — Metrics
# Keyword list đồng bộ Report 2 để clinical consistency có cùng định nghĩa với Stage 00.
KEYWORDS = [
    "hạch", "phổi", "xương", "thận", "gan", "cổ", "chậu", "đại tràng", "ổ bụng", "lách",
    "não", "amydal", "trung thất", "tuyến giáp", "dạ dày", "vú", "thượng đòn", "vòm", "tụy",
    "tuyến tiền liệt",
]
SUV_TERMS = ["suv", "suvmax", "suv max"]

def section_ok(t):
    u = str(t).upper()
    findings_ok = ("MÔ TẢ" in u) or ("MO TA" in u) or ("FINDINGS" in u)
    impression_ok = ("NHẬN ĐỊNH" in u) or ("NHAN DINH" in u) or ("KẾT QUẢ" in u) or ("KET QUA" in u) or ("IMPRESSION" in u)
    return findings_ok, impression_ok

def kw_set(t):
    t = str(t).lower()
    return {k for k in KEYWORDS if k in t}

def f1_sets(a, b):
    if not a and not b: return 1.,1.,1.
    if not a or not b:  return 0.,0.,0.
    tp=len(a&b); p=tp/max(len(a),1); r=tp/max(len(b),1)
    return p,r,2*p*r/max(p+r,1e-9)

preds = generated["generated_report"].fillna("").astype(str).tolist()
refs  = generated["reference_report"].fillna("").astype(str).tolist()
metrics: Dict[str,Any] = {"n_eval": len(generated)}

if hf_evaluate and len(generated):
    try:
        rouge = hf_evaluate.load("rouge")
        metrics.update({f"rouge_{k}":float(v) for k,v in rouge.compute(predictions=preds,references=refs).items()})
    except Exception as e: metrics["rouge_error"]=str(e)
    try:
        bleu = hf_evaluate.load("sacrebleu")
        metrics["sacrebleu"] = float(bleu.compute(predictions=preds,references=[[r] for r in refs])["score"])
    except Exception as e: metrics["bleu_error"]=str(e)

find_ok=[]; imp_ok=[]; f1s=[]
for pred,ref in zip(preds,refs):
    fo,io=section_ok(pred); find_ok.append(fo); imp_ok.append(io)
    _,_,f=f1_sets(kw_set(pred),kw_set(ref)); f1s.append(f)
metrics["section_findings_rate"]   = float(np.mean(find_ok))
metrics["section_impression_rate"] = float(np.mean(imp_ok))
metrics["keyword_f1_report2"]      = float(np.mean(f1s))
metrics["mean_pred_words"]         = float(np.mean([len(p.split()) for p in preds]))
metrics["mean_ref_words"]          = float(np.mean([len(r.split()) for r in refs]))
metrics["suv_mention_pred_rate"]   = float(np.mean([any(term in p.lower() for term in SUV_TERMS) for p in preds])) if preds else 0.0
metrics["suv_mention_ref_rate"]    = float(np.mean([any(term in r.lower() for term in SUV_TERMS) for r in refs])) if refs else 0.0

(OUTPUT_ROOT/"eval_metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(metrics,ensure_ascii=False,indent=2))

In [ ]:
# CELL 13 — Final check
print("===== OUTPUT FILES =====")
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p}  {p.stat().st_size:,} bytes")
print("\nDone — Save Version để giữ adapter + generated_reports + eval_metrics.")